In [ ]:
"""1. Hyperparameter optimization for Bayesian MTGNN model with plotting functionality (same as the baseline)."""
import argparse
import math
import time
import torch
import torch.nn as nn
from net import gtnet
import numpy as np
import importlib
import random
from util import *
from trainer import Optim
import sys
from random import randrange
from matplotlib import pyplot as plt
import time

plt.rcParams['savefig.dpi'] = 1200


def inverse_diff_2d(output, I,shift):
    output[0,:]=torch.exp(output[0,:]+torch.log(I+shift))-shift
    for i in range(1,output.shape[0]):
        output[i,:]= torch.exp(output[i,:]+torch.log(output[i-1,:]+shift))-shift
    return output

def inverse_diff_3d(output, I,shift):
    output[:,0,:]=torch.exp(output[:,0,:]+torch.log(I+shift))-shift
    for i in range(1,output.shape[1]):
        output[:,i,:]=torch.exp(output[:,i,:]+torch.log(output[:,i-1,:]+shift))-shift
    return output


def plot_data(data,title):
    x=range(1,len(data)+1)
    plt.plot(x,data,'b-',label='Actual')
    plt.legend(loc="best",prop={'size': 11})
    plt.axis('tight')
    plt.grid(True)
    plt.title(title, y=1.03,fontsize=18)
    plt.ylabel("Trend",fontsize=15)
    plt.xlabel("Month",fontsize=15)
    locs, labs = plt.xticks() 
    plt.xticks(rotation='vertical',fontsize=13) 
    plt.yticks(fontsize=13)
    fig = plt.gcf()
    plt.show()


# for figure display, we rename columns
def consistent_name(name):

    if name=='CAPTCHA' or name=='DNSSEC' or name=='RRAM':
        return name

    #e.g., University of london
    if not name.isupper():
        words=name.split(' ')
        result=''
        for i,word in enumerate(words):
            if len(word)<=2: #e.g., "of"
                result+=word
            else:
                result+=word[0].upper()+word[1:]
            
            if i<len(words)-1:
                result+=' '

        return result
    

    words= name.split(' ')
    result=''
    for i,word in enumerate(words):
        if len(word)<=3 or '/' in word or word=='MITM' or word =='SIEM':
            result+=word
        else:
            result+=word[0]+(word[1:].lower())
        
        if i<len(words)-1:
            result+=' '
        
    return result

#computes and saves validation/testing error to a text file given a single node's prediction and actual curve values
def save_metrics_1d(predict, test, title, type):
    #RRSE according to Lai et.al - numerator
    sum_squared_diff = torch.sum(torch.pow(test - predict, 2))
    root_sum_squared= math.sqrt(sum_squared_diff) #numerator

    #Relative Absolute Error RAE  - numerator
    sum_absolute_diff= torch.sum(torch.abs(test - predict))

    #RRSE according to Lai et.al - denominator
    test_s=test
    mean_all = torch.mean(test_s) # calculate the mean of each column in test
    diff_r = test_s - mean_all # subtract the mean from each element in the tensor test
    sum_squared_r = torch.sum(torch.pow(diff_r, 2))# square the result and sum over all elements
    root_sum_squared_r=math.sqrt(sum_squared_r)#denominator

    #RRSE according to Lai et.al
    rrse=root_sum_squared/root_sum_squared_r

    #Relative Absolute Error RAE - denominator
    sum_absolute_r=torch.sum(torch.abs(diff_r))# absolute the result and sum over all elements
    
    #Relative Absolute Error RAE
    rae=sum_absolute_diff/sum_absolute_r 
    rae=rae.item()


    title=title.replace('/','_')
    with open('model/Bayesian/'+type+'/'+title+'_'+type+'.txt',"w") as f:
      f.write('rse:'+str(rrse)+'\n')
      f.write('rae:'+str(rae)+'\n')
      f.close()


#plots predicted curve with actual curve. The x axis can be adjusted as needed
def plot_predicted_actual(predicted, actual, title, type,variance, confidence_95):

    #all months
    months=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    M=[]
    for year in range (11,25):   
        for month in months:
            if year==11 and month not in ['Jul','Aug','Sep','Oct','Nov','Dec']:
                continue
            M.append(month+'-'+str(year))   
    M2=[]
    p=[]
    
    #last 3 years
    if type=='Testing':
        M=M[-len(predicted):]
        for index,value in enumerate(M):
            if 'Dec' in M[index] or 'Mar' in M[index] or 'Jun' in M[index] or 'Sep' in M[index]:
                M2.append(M[index])
                p.append(index+1) 
    
    else: ##Validation x axis: Oct-16 to Sep-19
        M=M[63:99]
        for index,value in enumerate(M):
            if 'Dec' in M[index] or 'Mar' in M[index] or 'Jun' in M[index] or 'Sep' in M[index]:
                M2.append(M[index])
                p.append(index+1) 

    x=range(1,len(predicted)+1)
    plt.plot(x,actual,'b-',label='Actual')
    plt.plot(x,predicted,'--', color='purple',label='Predicted')
    # Plot the confidence interval as a shaded region
    plt.fill_between(x, predicted-confidence_95.numpy(), predicted+confidence_95.numpy(), alpha=0.5, color='pink', label='95% Confidence')
    plt.legend(loc="best",prop={'size': 11})
    plt.axis('tight')
    plt.grid(True)
    plt.title(title, y=1.03,fontsize=18)
    plt.ylabel("Trend",fontsize=15)
    plt.xlabel("Month",fontsize=15)
    locs, labs = plt.xticks() 
    plt.xticks(ticks = p ,labels = M2, rotation='vertical',fontsize=13) 
    plt.yticks(fontsize=13)
    fig = plt.gcf()
    title=title.replace('/','_')
    plt.savefig('model/Bayesian/'+type+'/'+title+'_'+type+'.png', bbox_inches="tight")
    plt.savefig('model/Bayesian/'+type+'/'+title+'_'+type+".pdf", bbox_inches = "tight", format='pdf')


    plt.show(block=False)
    plt.pause(2)
    plt.close()


#symmetric mean absolute percentage error (optional)
def s_mape(yTrue,yPred):
  mape=0
  for i in range(len(yTrue)):
    mape+= abs(yTrue[i]-yPred[i])/ (abs(yTrue[i])+abs(yPred[i]))
  mape/=len(yTrue)

  return mape

#for testing the model on unseen data, a sliding window can be used when the output period of the model is smaller than the target period to be forecasted.
#The sliding window uses the output from previous step as input of the next step.
#In this case, the window was not slided (we predicted 36 months and the model by default predicts 36 months)
def evaluate_sliding_window(data, test_window, model, evaluateL2, evaluateL1, n_input, is_plot):
    #model.eval()# To get Bayesian estimation, we must comment out this line
    total_loss = 0
    total_loss_l1 = 0
    n_samples = 0
    predict = None
    test = None
    variance=None
    confidence_95=None
    predictions = []
    sum_squared_diff=0
    sum_absolute_diff=0
    r=random.randint(0, 141)
    r=0 # we can choose any random node index for printing
    print('testing r=',str(r))
    scale = data.scale.expand(test_window.size(0), data.m) #scale will have the max of each column (142 max values)
    print('Test Window Feature:',test_window[:,r])
    
    x_input = test_window[0:n_input, :].clone() # Generate input sequence

    for i in range(n_input, test_window.shape[0],data.out_len):

        print('**************x_input*******************')
        print(x_input[:,r])#prints 1 random column in the sliding window
        print('**************-------*******************')

        X = torch.unsqueeze(x_input,dim=0)
        X = torch.unsqueeze(X,dim=1)
        X = X.transpose(2,3)
        X = X.to(torch.float)


        y_true = test_window[i: i+data.out_len,:].clone() 


        # Bayesian estimation
        num_runs = 10

        # Create a list to store the outputs
        outputs = []


        # Use model to predict next time step
        for _ in range(num_runs):
            with torch.no_grad():
                output = model(X)  
                y_pred = output[-1, :, :,-1].clone()
                #if this is the last predicted window and it exceeds the test window range
                if y_pred.shape[0]>y_true.shape[0]:
                    y_pred=y_pred[:-(y_pred.shape[0]-y_true.shape[0]),]
            outputs.append(y_pred)

        # Stack the outputs along a new dimension
        outputs = torch.stack(outputs)


        y_pred=torch.mean(outputs,dim=0)
        var = torch.var(outputs, dim=0)#variance
        std_dev = torch.std(outputs, dim=0)#standard deviation

        # Calculate 95% confidence interval
        z=1.96
        confidence=z*std_dev/torch.sqrt(torch.tensor(num_runs))



        #shift the sliding window
        if data.P<=data.out_len:
            x_input = y_pred[-data.P:].clone()
        else:
            x_input = torch.cat([x_input[ -(data.P-data.out_len):, :].clone(), y_pred.clone()], dim=0)


        print('----------------------------Predicted months',str(i-n_input+1),'to',str(i-n_input+data.out_len),'--------------------------------------------------')
        print(y_pred.shape,y_true.shape)
        y_pred_o=y_pred
        y_true_o=y_true
        for z in range(y_true.shape[0]):
            print(y_pred_o[z,r],y_true_o[z,r]) #only one col
        print('------------------------------------------------------------------------------------------------------------')


        if predict is None:
            predict = y_pred
            test = y_true
            variance=var
            confidence_95=confidence
        else:
            predict = torch.cat((predict, y_pred))
            test = torch.cat((test, y_true))
            variance=torch.cat((variance, var))
            confidence_95=torch.cat((confidence_95,confidence))


    scale = data.scale.expand(test.size(0), data.m) #scale will have the max of each column (142 max values)

    #inverse normalisation
    predict*=scale
    test*=scale
    variance*=scale
    confidence_95*=scale


    #Relative Squared Error RSE according to Lai et.al - numerator
    sum_squared_diff = torch.sum(torch.pow(test - predict, 2))
    #Relative Absolute Error RAE - numerator
    sum_absolute_diff= torch.sum(torch.abs(test - predict))# numerator


    #Root Relative Squared Error RRSE according to Lai et.al - numerator
    root_sum_squared= math.sqrt(sum_squared_diff) #numerator
    
    #Root Relative Squared Error RRSE according to Lai et.al - denominator
    test_s=test
    mean_all = torch.mean(test_s, dim=0) # calculate the mean of each column in test call it Yj-
    diff_r = test_s - mean_all.expand(test_s.size(0), data.m) # subtract the mean from each element in the tensor test
    sum_squared_r = torch.sum(torch.pow(diff_r, 2))# square the result and sum over all elements
    root_sum_squared_r=math.sqrt(sum_squared_r)#denominator

    #RRSE according to Lai et.al
    rrse=root_sum_squared/root_sum_squared_r
    print('rrse=',root_sum_squared,'/',root_sum_squared_r)

    #Relative Absolute Error RAE - denominator
    sum_absolute_r=torch.sum(torch.abs(diff_r))# absolute the result and sum over all elements - denominator
    #Relative Absolute Error RAE
    rae=sum_absolute_diff/sum_absolute_r 
    rae=rae.item()



    predict = predict.data.cpu().numpy()
    Ytest = test.data.cpu().numpy()
    sigma_p = (predict).std(axis=0)
    sigma_g = (Ytest).std(axis=0)
    mean_p = predict.mean(axis=0)
    mean_g = Ytest.mean(axis=0)
    index = (sigma_g != 0)
    correlation = ((predict - mean_p) * (Ytest - mean_g)).mean(axis=0) / (sigma_p * sigma_g) #Pearson's correlation coefficient?
    correlation = (correlation[index]).mean()

    #s-mape
    smape=0
    for z in range(Ytest.shape[1]):
        smape+=s_mape(Ytest[:,z],predict[:,z])
    smape/=Ytest.shape[1]

    #plot predicted vs actual and save errors to file
    counter=0
    if is_plot:
        for v in range(r,r+142):
            col=v%data.m
            
            node_name=DataLoaderS.col[col].replace('-ALL','').replace('Mentions-','Mentions of ').replace(' ALL','').replace('Solution_','').replace('_Mentions','')
            node_name=consistent_name(node_name)
            
            #save error to file
            save_metrics_1d(torch.from_numpy(predict[:,col]),torch.from_numpy(Ytest[:,col]),node_name,'Testing')
            #plot
            plot_predicted_actual(predict[:,col],Ytest[:,col],node_name, 'Testing',variance[:,col],confidence_95[:,col])
            counter+=1

    return rrse,rae,correlation, smape



def evaluate(data, X, Y, model, evaluateL2, evaluateL1, batch_size, is_plot):
    #model.eval()# To get Bayesian estimation, we must comment out this line
    total_loss = 0
    total_loss_l1 = 0
    n_samples = 0
    predict = None
    test = None
    variance=None
    confidence_95=None
    sum_squared_diff=0
    sum_absolute_diff=0
    r=0 #we choose any node index for printing (debugging)
    print('validation r=',str(r))

    for X, Y in data.get_batches(X, Y, batch_size, False):
        X = torch.unsqueeze(X,dim=1)
        X = X.transpose(2,3)

        # Bayesian estimation
        num_runs = 10

        # Create a list to store the outputs
        outputs = []

        # Run the model multiple times (10)
        with torch.no_grad():
            for _ in range(num_runs):
                output = model(X)
                output = torch.squeeze(output)
                if len(output.shape) == 1 or len(output.shape) == 2:
                    output = output.unsqueeze(dim=0)
                outputs.append(output)
            

        # Stack the outputs along a new dimension
        outputs = torch.stack(outputs)

        # Calculate mean, variance, and standard deviation
        mean = torch.mean(outputs, dim=0)
        var = torch.var(outputs, dim=0)#variance
        std_dev = torch.std(outputs, dim=0)#standard deviation

        # Calculate 95% confidence interval
        z=1.96
        confidence=z*std_dev/torch.sqrt(torch.tensor(num_runs))

        output=mean #we will consider the mean to be the prediction

        scale = data.scale.expand(Y.size(0), Y.size(1), data.m) #scale will have the max of each column (142 max values)
        
        #inverse normalisation
        output*=scale
        Y*=scale
        var*=scale
        confidence*=scale

        if predict is None:
            predict = output
            test = Y
            variance=var
            confidence_95=confidence
        else:
            predict = torch.cat((predict, output))
            test = torch.cat((test, Y))
            variance= torch.cat((variance, var))
            confidence_95=torch.cat((confidence_95,confidence))


        print('EVALUATE RESULTS:')
        scale = data.scale.expand(Y.size(0), Y.size(1), data.m) #scale will have the max of each column (142 max values)
        y_pred_o=output
        y_true_o=Y
        for z in range(Y.shape[1]):
            print(y_pred_o[0,z,r],y_true_o[0,z,r]) #only one col
        
        total_loss += evaluateL2(output, Y).item()
        total_loss_l1 += evaluateL1(output, Y).item()
        n_samples += (output.size(0) * output.size(1) * data.m)

        #RRSE according to Lai et.al
        sum_squared_diff += torch.sum(torch.pow(Y - output, 2))
        #Relative Absolute Error RAE - numerator
        sum_absolute_diff+=torch.sum(torch.abs(Y - output))

    #The below 2 lines are not used
    rse = math.sqrt(total_loss / n_samples) / data.rse 
    rae = (total_loss_l1 / n_samples) / data.rae 

    #RRSE according to Lai et.al - numerator
    root_sum_squared= math.sqrt(sum_squared_diff) #numerator
    
    #RRSE according to Lai et.al - denominator
    test_s=test
    mean_all = torch.mean(test_s, dim=(0,1)) # calculate the mean of each column in test
    diff_r = test_s - mean_all.expand(test_s.size(0), test_s.size(1), data.m) # subtract the mean from each element in the tensor test
    sum_squared_r = torch.sum(torch.pow(diff_r, 2))# square the result and sum over all elements
    root_sum_squared_r=math.sqrt(sum_squared_r)#denominator

    #RRSE according to Lai et.al
    rrse=root_sum_squared/root_sum_squared_r #RRSE

    #Relative Absolute Error RAE
    sum_absolute_r=torch.sum(torch.abs(diff_r))# absolute the result and sum over all elements - denominator
    rae=sum_absolute_diff/sum_absolute_r # RAE
    rae=rae.item()


    predict = predict.data.cpu().numpy()
    Ytest = test.data.cpu().numpy()
    sigma_p = (predict).std(axis=0)
    sigma_g = (Ytest).std(axis=0)
    mean_p = predict.mean(axis=0)
    mean_g = Ytest.mean(axis=0)
    index = (sigma_g != 0)
    correlation = ((predict - mean_p) * (Ytest - mean_g)).mean(axis=0)/ (sigma_p * sigma_g) #Pearson's correlation coefficient?
    correlation = (correlation[index]).mean()

    #s-mape
    smape=0
    for x in range(Ytest.shape[0]):
        for z in range(Ytest.shape[2]):
            smape+=s_mape(Ytest[x,:,z],predict[x,:,z])
    smape/=Ytest.shape[0]*Ytest.shape[2]


    #plot actual vs predicted curves and save errors to file
    counter=0
    if is_plot:
        for v in range(r,r+142):
            col=v%data.m
            node_name=DataLoaderS.col[col].replace('-ALL','').replace('Mentions-','Mentions of ').replace(' ALL','').replace('Solution_','').replace('_Mentions','')
            node_name=consistent_name(node_name)
            save_metrics_1d(torch.from_numpy(predict[-1,:,col]),torch.from_numpy(Ytest[-1,:,col]),node_name,'Validation')
            plot_predicted_actual(predict[-1,:,col],Ytest[-1,:,col],node_name, 'Validation', variance[-1,:,col], confidence_95[-1,:,col])
            counter+=1
    return rrse, rae, correlation, smape


def train(data, X, Y, model, criterion, optim, batch_size):
    model.train()
    total_loss = 0
    n_samples = 0
    iter = 0

    for X, Y in data.get_batches(X, Y, batch_size, True):
        model.zero_grad()
        X = torch.unsqueeze(X,dim=1)
        X = X.transpose(2,3)
        if iter % args.step_size == 0:
            perm = np.random.permutation(range(args.num_nodes))
        num_sub = int(args.num_nodes / args.num_split)

        for j in range(args.num_split):
            if j != args.num_split - 1:
                id = perm[j * num_sub:(j + 1) * num_sub]
            else:
                id = perm[j * num_sub:]

            id = torch.tensor(id).to(device)
            tx = X[:, :, :, :] 
            ty = Y[:, :, :] 
            output = model(tx)           
            output = torch.squeeze(output,3)
            scale = data.scale.expand(output.size(0), output.size(1), data.m)
            scale = scale[:,:,:] 
            
            output*=scale 
            ty*=scale


            loss = criterion(output, ty)
            loss.backward()
            total_loss += loss.item()
            n_samples += (output.size(0) * output.size(1) * data.m)
            
            grad_norm = optim.step()

        if iter%1==0:
            print('iter:{:3d} | loss: {:.3f}'.format(iter,loss.item()/(output.size(0) * output.size(1)* data.m)))
        iter += 1
    return total_loss / n_samples


parser = argparse.ArgumentParser(description='PyTorch Time series forecasting')
parser.add_argument('--data', type=str, default='./data/sm_data.txt',
                    help='location of the data file')
parser.add_argument('--log_interval', type=int, default=2000, metavar='N',
                    help='report interval')
parser.add_argument('--save', type=str, default='model/Bayesian/model.pt',
                    help='path to save the final model')
parser.add_argument('--optim', type=str, default='adam')
parser.add_argument('--L1Loss', type=bool, default=True)
parser.add_argument('--normalize', type=int, default=2)
parser.add_argument('--device',type=str,default='cuda:1',help='')
parser.add_argument('--gcn_true', type=bool, default=True, help='whether to add graph convolution layer')
parser.add_argument('--buildA_true', type=bool, default=True, help='whether to construct adaptive adjacency matrix')
parser.add_argument('--gcn_depth',type=int,default=2,help='graph convolution depth')
parser.add_argument('--num_nodes',type=int,default=142,help='number of nodes/variables')
parser.add_argument('--dropout',type=float,default=0.3,help='dropout rate')
parser.add_argument('--subgraph_size',type=int,default=20,help='k')
parser.add_argument('--node_dim',type=int,default=40,help='dim of nodes')
parser.add_argument('--dilation_exponential',type=int,default=2,help='dilation exponential')
parser.add_argument('--conv_channels',type=int,default=16,help='convolution channels')
parser.add_argument('--residual_channels',type=int,default=16,help='residual channels')
parser.add_argument('--skip_channels',type=int,default=32,help='skip channels')
parser.add_argument('--end_channels',type=int,default=64,help='end channels')
parser.add_argument('--in_dim',type=int,default=1,help='inputs dimension')
parser.add_argument('--seq_in_len',type=int,default=10,help='input sequence length')
parser.add_argument('--seq_out_len',type=int,default=36,help='output sequence length')
parser.add_argument('--horizon', type=int, default=1) 
parser.add_argument('--layers',type=int,default=5,help='number of layers')

parser.add_argument('--batch_size',type=int,default=8,help='batch size')
parser.add_argument('--lr',type=float,default=0.001,help='learning rate')
parser.add_argument('--weight_decay',type=float,default=0.00001,help='weight decay rate')

parser.add_argument('--clip',type=int,default=10,help='clip')

parser.add_argument('--propalpha',type=float,default=0.05,help='prop alpha')
parser.add_argument('--tanhalpha',type=float,default=3,help='tanh alpha')

parser.add_argument('--epochs',type=int,default=200,help='')
parser.add_argument('--num_split',type=int,default=1,help='number of splits for graphs')
parser.add_argument('--step_size',type=int,default=100,help='step_size')


args = parser.parse_args()
device = torch.device('cpu')
torch.set_num_threads(3)

def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fixed_seed = 123

def main(experiment):
    # Set fixed random seed for reproducibility
    set_random_seed(fixed_seed)

    #model hyper-parameters
    gcn_depths=[1,2,3]
    lrs=[0.01,0.001,0.0005,0.0008,0.0001,0.0003,0.005]#[0.00001,0.0001,0.0002,0.0003]
    convs=[4,8,16]
    ress=[16,32,64]
    skips=[64,128,256]
    ends=[256,512,1024]
    layers=[1,2]
    ks=[20,30,40,50,60,70,80,90,100]
    dropouts=[0.2,0.3,0.4,0.5,0.6,0.7]
    dilation_exs=[1,2,3]
    node_dims=[20,30,40,50,60,70,80,90,100]
    prop_alphas=[0.05,0.1,0.15,0.2,0.3,0.4,0.6,0.8]
    tanh_alphas=[0.05,0.1,0.5,1,2,3,5,7,9]


    best_val = 10000000
    best_rse=  10000000
    best_rae=  10000000
    best_corr= -10000000
    best_smape=10000000
    
    best_test_rse=10000000
    best_test_corr=-10000000

    best_hp=[]


    #random search
    for q in range(60):

        #hps
        gcn_depth=gcn_depths[randrange(len(gcn_depths))]
        lr=lrs[randrange(len(lrs))]
        conv=convs[randrange(len(convs))]
        res=ress[randrange(len(ress))]
        skip=skips[randrange(len(skips))]
        end=ends[randrange(len(ends))]
        layer=layers[randrange(len(layers))]
        k=ks[randrange(len(ks))]
        dropout=dropouts[randrange(len(dropouts))]
        dilation_ex=dilation_exs[randrange(len(dilation_exs))]
        node_dim=node_dims[randrange(len(node_dims))]
        prop_alpha=prop_alphas[randrange(len(prop_alphas))]
        tanh_alpha=tanh_alphas[randrange(len(tanh_alphas))]
        

        Data = DataLoaderS(args.data, 0.43, 0.30, device, args.horizon, args.seq_in_len, args.normalize,args.seq_out_len)
    

        print('train X:',Data.train[0].shape)
        print('train Y:', Data.train[1].shape)
        print('valid X:',Data.valid[0].shape)
        print('valid Y:',Data.valid[1].shape)
        print('test X:',Data.test[0].shape)
        print('test Y:',Data.test[1].shape)
        print('test window:', Data.test_window.shape)

        print('length of training set=',Data.train[0].shape[0])
        print('length of validation set=',Data.valid[0].shape[0])
        print('length of testing set=',Data.test[0].shape[0])
        print('valid=',int((0.43 + 0.3) * Data.n))
        
       
        
        model = gtnet(args.gcn_true, args.buildA_true, gcn_depth, args.num_nodes,
                    device, Data.adj, dropout=dropout, subgraph_size=k,
                    node_dim=node_dim, dilation_exponential=dilation_ex,
                    conv_channels=conv, residual_channels=res,
                    skip_channels=skip, end_channels= end,
                    seq_length=args.seq_in_len, in_dim=args.in_dim, out_dim=args.seq_out_len,
                    layers=layer, propalpha=prop_alpha, tanhalpha=tanh_alpha, layer_norm_affline=False)
        

        print(args)
        print('The recpetive field size is', model.receptive_field)
        nParams = sum([p.nelement() for p in model.parameters()])
        print('Number of model parameters is', nParams, flush=True)

        if args.L1Loss:
            criterion = nn.L1Loss(reduction='sum').to(device)
        else:
            criterion = nn.MSELoss(reduction='sum').to(device)
        evaluateL2 = nn.MSELoss(reduction='sum').to(device) #MSE
        evaluateL1 = nn.L1Loss(reduction='sum').to(device) #MAE

        optim = Optim(
            model.parameters(), args.optim, lr, args.clip, lr_decay=args.weight_decay
        )
        
        es_counter=0 #early stopping
        # At any point you can hit Ctrl + C to break out of training early.
        try:
            print('begin training')
            for epoch in range(1, args.epochs + 1):
                print('Experiment:',(experiment+1))
                print('Iter:',q)
                print('epoch:',epoch)
                print('hp=',[gcn_depth,lr,conv,res,skip,end, k, dropout, dilation_ex, node_dim, prop_alpha, tanh_alpha, layer, epoch])
                print('best sum=',best_val)
                print('best rrse=',best_rse)
                print('best rrae=',best_rae)
                print('best corr=',best_corr)
                print('best smape=',best_smape)       
                print('best hps=',best_hp)
                print('best test rse=',best_test_rse)
                print('best test corr=',best_test_corr)

                
                es_counter+=1 # feel free to use this for early stopping (not used)

                epoch_start_time = time.time()
                train_loss = train(Data, Data.train[0], Data.train[1], model, criterion, optim, args.batch_size)
                val_loss, val_rae, val_corr, val_smape = evaluate(Data, Data.valid[0], Data.valid[1], model, evaluateL2, evaluateL1,
                                                 args.batch_size,False)
                print(
                    '| end of epoch {:3d} | time: {:5.2f}s | train_loss {:5.4f} | valid rse {:5.4f} | valid rae {:5.4f} | valid corr  {:5.4f} | valid smape  {:5.4f}'.format(
                        epoch, (time.time() - epoch_start_time), train_loss, val_loss, val_rae, val_corr, val_smape), flush=True)
                # Save the model if the validation loss is the best we've seen so far.
                sum_loss=val_loss+val_rae-val_corr
                if (not math.isnan(val_corr)) and val_loss < best_rse:
                    with open(args.save, 'wb') as f:
                        torch.save(model, f)
                    best_val = sum_loss
                    best_rse= val_loss
                    best_rae= val_rae
                    best_corr= val_corr
                    best_smape=val_smape

                    best_hp=[gcn_depth,lr,conv,res,skip,end, k, dropout, dilation_ex, node_dim, prop_alpha, tanh_alpha, layer, epoch]
                    
                    es_counter=0
                    
                    test_acc, test_rae, test_corr, test_smape = evaluate_sliding_window(Data, Data.test_window, model, evaluateL2, evaluateL1,
                                           args.seq_in_len, False) 
                    print('********************************************************************************************************')
                    print("test rse {:5.4f} | test rae {:5.4f} | test corr {:5.4f}| test smape {:5.4f}".format(test_acc, test_rae, test_corr, test_smape), flush=True)
                    print('********************************************************************************************************')
                    best_test_rse=test_acc
                    best_test_corr=test_corr

        except KeyboardInterrupt:
            print('-' * 89)
            print('Exiting from training early')

    print('best val loss=',best_val)
    print('best hps=',best_hp)
    #save best hp to desk
    with open('model/Bayesian/hp.txt',"w") as f:
        f.write(str(best_hp))
        f.close()
    # Load the best saved model.
    with open(args.save, 'rb') as f:
        model = torch.load(f)

    vtest_acc, vtest_rae, vtest_corr, vtest_smape = evaluate(Data, Data.valid[0], Data.valid[1], model, evaluateL2, evaluateL1,
                                         args.batch_size, True)

    test_acc, test_rae, test_corr, test_smape = evaluate_sliding_window(Data, Data.test_window, model, evaluateL2, evaluateL1,
                                         args.seq_in_len, True) 
    print('********************************************************************************************************')    
    print("final test rse {:5.4f} | test rae {:5.4f} | test corr {:5.4f} | test smape {:5.4f}".format(test_acc, test_rae, test_corr, test_smape))
    print('********************************************************************************************************')
    return vtest_acc, vtest_rae, vtest_corr, vtest_smape, test_acc, test_rae, test_corr, test_smape

if __name__ == "__main__":
    vacc = []
    vrae = []
    vcorr = []
    vsmape=[]
    acc = []
    rae = []
    corr = []
    smape=[]
    for i in range(1):
        val_acc, val_rae, val_corr, val_smape, test_acc, test_rae, test_corr, test_smape = main(i)
        vacc.append(val_acc)
        vrae.append(val_rae)
        vcorr.append(val_corr)
        vsmape.append(val_smape)
        acc.append(test_acc)
        rae.append(test_rae)
        corr.append(test_corr)
        smape.append(test_smape)
    print('\n\n')
    print('1 run average')
    print('\n\n')
    print("valid\trse\trae")
    print("mean\t{:5.4f}\t{:5.4f}".format(np.mean(vacc), np.mean(vrae)))
    print("std\t{:5.4f}\t{:5.4f}".format(np.std(vacc), np.std(vrae)))
    print('\n\n')
    print("test\trse\trae")
    print("mean\t{:5.4f}\t{:5.4f}".format(np.mean(acc), np.mean(rae)))
    print("std\t{:5.4f}\t{:5.4f}".format(np.std(acc), np.std(rae)))



In [ ]:
"""2. Script to train the final model on the full data, utilising the optimal set of hyper-parameters found in the file train_test.py"""
import ast
import argparse
import math
import time
import torch
import torch.nn as nn
from net import gtnet
import numpy as np
import importlib
import random
from o_util import *
from trainer import Optim
import sys
from random import randrange
from matplotlib import pyplot as plt
import time

# This script trains the final model on the full data, utilising the optimal set of hyper-parameters found in the file train_test


plt.rcParams['savefig.dpi'] = 1200

def train(data, X, Y, model, criterion, optim, batch_size):
    model.train()
    total_loss = 0
    n_samples = 0
    iter = 0

    for X, Y in data.get_batches(X, Y, batch_size, True):
        model.zero_grad()
        X = torch.unsqueeze(X,dim=1)
        X = X.transpose(2,3)
        if iter % args.step_size == 0:
            perm = np.random.permutation(range(args.num_nodes))
        num_sub = int(args.num_nodes / args.num_split)

        for j in range(args.num_split):
            if j != args.num_split - 1:
                id = perm[j * num_sub:(j + 1) * num_sub]
            else:
                id = perm[j * num_sub:]

            id = torch.tensor(id).to(device)
            tx = X[:, :, :, :] 
            ty = Y[:, :, :] 
            output = model(tx)         
            output = torch.squeeze(output,3)
            scale = data.scale.expand(output.size(0), output.size(1), data.m)
            scale = scale[:,:,:] 
            
            output*=scale 
            ty*=scale

            loss = criterion(output, ty)
            loss.backward()
            total_loss += loss.item()
            n_samples += (output.size(0) * output.size(1) * data.m)
            
            grad_norm = optim.step()

        if iter%1==0:
            print('iter:{:3d} | loss: {:.3f}'.format(iter,loss.item()/(output.size(0) * output.size(1)* data.m)))
        iter += 1
    return total_loss / n_samples


parser = argparse.ArgumentParser(description='PyTorch Time series forecasting')
parser.add_argument('--data', type=str, default='./data/sm_data.txt',
                    help='location of the data file')
parser.add_argument('--log_interval', type=int, default=2000, metavar='N',
                    help='report interval')
parser.add_argument('--save', type=str, default='model/Bayesian/o_model.pt',
                    help='path to save the final model')
parser.add_argument('--optim', type=str, default='adam')
parser.add_argument('--L1Loss', type=bool, default=True)
parser.add_argument('--normalize', type=int, default=2)
parser.add_argument('--device',type=str,default='cuda:1',help='')
parser.add_argument('--gcn_true', type=bool, default=True, help='whether to add graph convolution layer')
parser.add_argument('--buildA_true', type=bool, default=True, help='whether to construct adaptive adjacency matrix')
parser.add_argument('--gcn_depth',type=int,default=2,help='graph convolution depth')
parser.add_argument('--num_nodes',type=int,default=142,help='number of nodes/variables')
parser.add_argument('--dropout',type=float,default=0.3,help='dropout rate')
parser.add_argument('--subgraph_size',type=int,default=20,help='k')
parser.add_argument('--node_dim',type=int,default=40,help='dim of nodes')
parser.add_argument('--dilation_exponential',type=int,default=2,help='dilation exponential')
parser.add_argument('--conv_channels',type=int,default=16,help='convolution channels')
parser.add_argument('--residual_channels',type=int,default=16,help='residual channels')
parser.add_argument('--skip_channels',type=int,default=32,help='skip channels')
parser.add_argument('--end_channels',type=int,default=64,help='end channels')
parser.add_argument('--in_dim',type=int,default=1,help='inputs dimension')
parser.add_argument('--seq_in_len',type=int,default=10,help='input sequence length')
parser.add_argument('--seq_out_len',type=int,default=36,help='output sequence length')
parser.add_argument('--horizon', type=int, default=1) 
parser.add_argument('--layers',type=int,default=5,help='number of layers')
parser.add_argument('--batch_size',type=int,default=8,help='batch size')
parser.add_argument('--lr',type=float,default=0.001,help='learning rate')
parser.add_argument('--weight_decay',type=float,default=0.00001,help='weight decay rate')
parser.add_argument('--clip',type=int,default=10,help='clip')
parser.add_argument('--propalpha',type=float,default=0.05,help='prop alpha')
parser.add_argument('--tanhalpha',type=float,default=3,help='tanh alpha')
parser.add_argument('--epochs',type=int,default=200,help='')
parser.add_argument('--num_split',type=int,default=1,help='number of splits for graphs')
parser.add_argument('--step_size',type=int,default=100,help='step_size')


args = parser.parse_args()
device = torch.device('cpu')
torch.set_num_threads(3)

def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fixed_seed = 123
set_random_seed(fixed_seed)


#read hyper-parameters
filename = "model/Bayesian/hp.txt"
with open(filename, 'r') as file:
    content = file.read()
    hp = ast.literal_eval(content)

print('hp',hp)

#train the model
gcn_depth=hp[0]
lr=hp[1]
conv=hp[2]
res=hp[3]
skip=hp[4]
end=hp[5]
layer=hp[-2]
k=hp[6]
dropout=hp[7]
dilation_ex=hp[8]
node_dim=hp[9]
prop_alpha=hp[10]
tanh_alpha=hp[11]
epochs=hp[-1]


Data = DataLoaderS(args.data, 0.43, 0.30, device, args.horizon, args.seq_in_len, args.normalize,args.seq_out_len)



model = gtnet(args.gcn_true, args.buildA_true, gcn_depth, args.num_nodes,
            device, Data.adj, dropout=dropout, subgraph_size=k,
            node_dim=node_dim, dilation_exponential=dilation_ex,
            conv_channels=conv, residual_channels=res,
            skip_channels=skip, end_channels= end,
            seq_length=args.seq_in_len, in_dim=args.in_dim, out_dim=args.seq_out_len,
            layers=layer, propalpha=prop_alpha, tanhalpha=tanh_alpha, layer_norm_affline=False)



print(args)
print('The recpetive field size is', model.receptive_field)
nParams = sum([p.nelement() for p in model.parameters()])
print('Number of model parameters is', nParams, flush=True)

if args.L1Loss:
    criterion = nn.L1Loss(reduction='sum').to(device)
else:
    criterion = nn.MSELoss(reduction='sum').to(device)
evaluateL2 = nn.MSELoss(reduction='sum').to(device) #MSE
evaluateL1 = nn.L1Loss(reduction='sum').to(device) #MAE


optim = Optim(
    model.parameters(), args.optim, lr, args.clip, lr_decay=args.weight_decay
)

# At any point you can hit Ctrl + C to break out of training early.
try:
    print('begin training')
    for epoch in range(1, epochs + 1):
        print('epoch:',epoch)
        epoch_start_time = time.time()
        train_loss = train(Data, Data.train[0], Data.train[1], model, criterion, optim, args.batch_size)
    with open(args.save, 'wb') as f:
        torch.save(model, f)        
except KeyboardInterrupt:
    print('-' * 89)
    print('Exiting from training early')

In [ ]:
"""3. Script to evaluate the final model on the test data, and return plots of evaluation results along with error metrics."""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
from scipy import stats
from scipy.spatial.distance import jensenshannon
from scipy.stats import ks_2samp, wasserstein_distance
from scipy.stats import norm
import json
import time
import warnings
from datetime import datetime, timedelta
import os
import sys
import ast
from collections import defaultdict, Counter

# Import the BMTGNN model architecture
try:
    from net import gtnet
    from o_util import DataLoaderS
except ImportError as e:
    print(f"Warning: Could not import BMTGNN dependencies: {e}")
    print("Make sure net.py and o_util.py are in your Python path")

warnings.filterwarnings('ignore')

class BMTGNNEvaluator:
    def __init__(self, model_path, data_file, config_file=None):
        self.model_path = model_path
        self.data_file = data_file
        self.config_file = config_file
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Load default configuration similar to train.py
        self.config = self.get_default_config()
        
        # Load model and configuration
        self.load_model()
        
        # Metrics storage
        self.metrics = {}
        
    def get_default_config(self):
        """Get default configuration matching train.py"""
        return {
            'data': './data/sm_data.txt',
            'normalize': 2,
            'device': 'cpu',
            'gcn_true': True,
            'buildA_true': True,
            'gcn_depth': 2,
            'num_nodes': 142,
            'dropout': 0.3,
            'subgraph_size': 20,
            'node_dim': 40,
            'dilation_exponential': 2,
            'conv_channels': 16,
            'residual_channels': 16,
            'skip_channels': 32,
            'end_channels': 64,
            'in_dim': 1,
            'seq_in_len': 10,
            'seq_out_len': 36,
            'horizon': 1,
            'layers': 5,
            'batch_size': 8,
            'propalpha': 0.05,
            'tanhalpha': 3,
            'num_split': 1,
            'step_size': 100
        }
    
    def load_hyperparameters(self):
        """Load hyperparameters from hp.txt file as in train.py"""
        if self.config_file and os.path.exists(self.config_file):
            try:
                with open(self.config_file, 'r') as file:
                    content = file.read()
                    hp = ast.literal_eval(content)
                print(f"Loaded hyperparameters: {hp}")
                return hp
            except Exception as e:
                print(f"Error loading hyperparameters: {e}")
        
        # Return default hyperparameters if file not found
        return [2, 0.001, 16, 16, 32, 64, 20, 0.3, 2, 40, 0.05, 3, 5, 200]
    
    def load_model(self):
        """Load the trained BMTGNN model - handles both state_dict and full model saves"""
        try:
            # Load hyperparameters first
            hp = self.load_hyperparameters()
            
            # Extract hyperparameters as in train.py
            gcn_depth = hp[0]
            lr = hp[1]
            conv = hp[2]
            res = hp[3]
            skip = hp[4]
            end = hp[5]
            layer = hp[-2]
            k = hp[6]
            dropout = hp[7]
            dilation_ex = hp[8]
            node_dim = hp[9]
            prop_alpha = hp[10]
            tanh_alpha = hp[11]
            epochs = hp[-1]
            
            # Update config with loaded hyperparameters
            self.config.update({
                'gcn_depth': gcn_depth,
                'dropout': dropout,
                'subgraph_size': k,
                'node_dim': node_dim,
                'dilation_exponential': dilation_ex,
                'conv_channels': conv,
                'residual_channels': res,
                'skip_channels': skip,
                'end_channels': end,
                'layers': layer,
                'propalpha': prop_alpha,
                'tanhalpha': tanh_alpha
            })
            
            print("BMTGNN Model Configuration:")
            for key, value in self.config.items():
                print(f"  {key}: {value}")
            
            # Load data to get adjacency matrix
            data_loader = DataLoaderS(self.data_file, 0.43, 0.30, self.device, 
                                    self.config['horizon'], self.config['seq_in_len'], 
                                    self.config['normalize'], self.config['seq_out_len'])
            
            # Create model
            self.model = gtnet(
                self.config['gcn_true'], 
                self.config['buildA_true'], 
                self.config['gcn_depth'], 
                self.config['num_nodes'],
                self.device, 
                data_loader.adj, 
                dropout=self.config['dropout'], 
                subgraph_size=self.config['subgraph_size'],
                node_dim=self.config['node_dim'], 
                dilation_exponential=self.config['dilation_exponential'],
                conv_channels=self.config['conv_channels'], 
                residual_channels=self.config['residual_channels'],
                skip_channels=self.config['skip_channels'], 
                end_channels=self.config['end_channels'],
                seq_length=self.config['seq_in_len'], 
                in_dim=self.config['in_dim'], 
                out_dim=self.config['seq_out_len'],
                layers=self.config['layers'], 
                propalpha=self.config['propalpha'], 
                tanhalpha=self.config['tanhalpha'], 
                layer_norm_affline=False
            ).to(self.device)
            
            # Load trained weights - handle both state_dict and full model
            print(f"Loading model from: {self.model_path}")
            checkpoint = torch.load(self.model_path, map_location=self.device)
            
            if isinstance(checkpoint, dict):
                if 'model_state_dict' in checkpoint:
                    # Standard state dict format
                    self.model.load_state_dict(checkpoint['model_state_dict'])
                    print("Loaded model from state_dict with 'model_state_dict' key")
                elif 'state_dict' in checkpoint:
                    # Alternative state dict format
                    self.model.load_state_dict(checkpoint['state_dict'])
                    print("Loaded model from state_dict with 'state_dict' key")
                else:
                    # Try to load as direct state dict
                    try:
                        self.model.load_state_dict(checkpoint)
                        print("Loaded model from direct state_dict")
                    except:
                        print("Could not load as state_dict, checking for full model...")
                        # Check if any values in the dict are model parameters
                        for key, value in checkpoint.items():
                            if hasattr(value, 'parameters'):
                                self.model = value
                                print(f"Loaded full model from key: {key}")
                                break
            else:
                # Check if it's a full model object
                if hasattr(checkpoint, 'parameters'):
                    self.model = checkpoint
                    print("Loaded full model object directly")
                else:
                    # Last attempt: try to load as state_dict
                    try:
                        self.model.load_state_dict(checkpoint)
                        print("Loaded model from checkpoint as state_dict")
                    except Exception as e:
                        print(f"Failed to load model: {e}")
                        raise
                
            self.model.eval()
            print("BMTGNN model loaded successfully")
            
            # Store data loader for later use
            self.data_loader = data_loader
            
            # Print model info
            nParams = sum([p.nelement() for p in self.model.parameters()])
            print(f'Number of model parameters: {nParams}')
            print(f'Receptive field size: {self.model.receptive_field}')
            
        except Exception as e:
            print(f"Error loading BMTGNN model: {e}")
            print("Attempting alternative loading strategy...")
            self.load_model_alternative()
    
    def load_model_alternative(self):
        """Alternative loading strategy for problematic model files"""
        try:
            # Try to load the entire model directly
            self.model = torch.load(self.model_path, weights_only=False, map_location=self.device)
            
            if hasattr(self.model, 'eval'):
                self.model.eval()
                print("Loaded model directly using torch.load()")
                
                # Move to device
                self.model = self.model.to(self.device)
                
                # Create a minimal data loader just for evaluation
                self.data_loader = DataLoaderS(self.data_file, 0.43, 0.30, self.device, 
                                            self.config['horizon'], self.config['seq_in_len'], 
                                            self.config['normalize'], self.config['seq_out_len'])
                return
            else:
                raise ValueError("Loaded object is not a valid model")
                
        except Exception as e:
            print(f"Alternative loading also failed: {e}")
            raise
    
    def generate_predictions(self, data_type='test', num_runs=10):
        """Generate predictions using BMTGNN model with Bayesian estimation"""
        print(f"Generating {data_type} predictions...")
        
        # Select data based on type
        if data_type == 'train':
            X, Y = self.data_loader.train
        elif data_type == 'valid':
            X, Y = self.data_loader.valid
        else:  # test
            X, Y = self.data_loader.test
        
        predictions = []
        actuals = []
        variances = []
        confidence_intervals = []
        all_outputs = []  # Store all runs for quantile calculation
        
        with torch.no_grad():
            # Get batches
            batches = list(self.data_loader.get_batches(X, Y, self.config['batch_size'], False))
            print(f"Processing {len(batches)} batches for {data_type} data")
            
            for batch_idx, (batch_X, batch_Y) in enumerate(batches):
                if batch_idx % 10 == 0:
                    print(f"Processing batch {batch_idx + 1}/{len(batches)}")
                
                batch_X = torch.unsqueeze(batch_X, dim=1)
                batch_X = batch_X.transpose(2, 3)
                
                # Bayesian estimation with multiple runs
                outputs = []
                for run_idx in range(num_runs):
                    output = self.model(batch_X)
                    output = torch.squeeze(output, 3)
                    outputs.append(output)
                
                # Stack outputs
                outputs = torch.stack(outputs)
                all_outputs.append(outputs.cpu().numpy())
                
                # Calculate statistics
                mean = torch.mean(outputs, dim=0)
                var = torch.var(outputs, dim=0)
                std_dev = torch.std(outputs, dim=0)
                
                # 95% confidence interval
                z = 1.96
                confidence = z * std_dev / torch.sqrt(torch.tensor(num_runs))
                
                # Apply scaling
                scale = self.data_loader.scale.expand(mean.size(0), mean.size(1), self.data_loader.m)
                mean_scaled = mean * scale
                batch_Y_scaled = batch_Y * scale
                var_scaled = var * scale
                confidence_scaled = confidence * scale
                
                predictions.append(mean_scaled.cpu().numpy())
                actuals.append(batch_Y_scaled.cpu().numpy())
                variances.append(var_scaled.cpu().numpy())
                confidence_intervals.append(confidence_scaled.cpu().numpy())
        
        # Concatenate all batches
        if predictions:
            predictions_array = np.concatenate(predictions, axis=0)
            actuals_array = np.concatenate(actuals, axis=0)
            variances_array = np.concatenate(variances, axis=0)
            confidence_array = np.concatenate(confidence_intervals, axis=0)
            all_outputs_array = np.concatenate(all_outputs, axis=1)  # Shape: (num_runs, total_samples, seq_len, num_nodes)
            
            print(f"Generated {predictions_array.shape[0]} prediction sequences for {data_type} data")
            return predictions_array, actuals_array, variances_array, confidence_array, all_outputs_array
        else:
            print(f"No predictions generated for {data_type} data")
            return np.array([]), np.array([]), np.array([]), np.array([]), np.array([])
    
    def calculate_quantile_loss(self, all_outputs, actuals, quantiles=[0.1, 0.5, 0.9]):
        """Calculate Quantile Loss (Pinball Loss) using multiple runs"""
        if all_outputs.size == 0 or actuals.size == 0:
            return {f'quantile_{int(q*100)}': float('inf') for q in quantiles}
            
        losses = {}
        
        # all_outputs shape: (num_runs, total_samples, seq_len, num_nodes)
        # actuals shape: (total_samples, seq_len, num_nodes)
        
        for q in quantiles:
            total_loss = 0
            total_count = 0
            
            # Calculate quantile across runs for each sample
            for sample_idx in range(all_outputs.shape[1]):
                sample_predictions = all_outputs[:, sample_idx, :, :]  # (num_runs, seq_len, num_nodes)
                
                # Calculate empirical quantile
                quantile_pred = np.percentile(sample_predictions, q * 100, axis=0)
                
                # Calculate pinball loss
                error = actuals[sample_idx] - quantile_pred
                loss = np.maximum(q * error, (q - 1) * error)
                total_loss += np.sum(loss)
                total_count += np.prod(loss.shape)
            
            losses[f'quantile_{int(q*100)}'] = total_loss / total_count if total_count > 0 else float('inf')
        
        return losses
    
    def calculate_ks_test(self, predictions, actuals):
        """Calculate Kolmogorov-Smirnov test statistics"""
        if predictions.size == 0 or actuals.size == 0:
            return 1.0, 0.0
            
        # Flatten for overall distribution comparison
        pred_flat = predictions.flatten()
        actual_flat = actuals.flatten()
        
        # Remove any NaN or Inf values
        pred_flat = pred_flat[np.isfinite(pred_flat)]
        actual_flat = actual_flat[np.isfinite(actual_flat)]
        
        if len(pred_flat) == 0 or len(actual_flat) == 0:
            return 1.0, 0.0
            
        stat, p_value = ks_2samp(pred_flat, actual_flat)
        return stat, p_value
    
    def calculate_js_divergence(self, predictions, actuals, bins=50):
        """Calculate Jensen-Shannon Divergence"""
        if predictions.size == 0 or actuals.size == 0:
            return 1.0
            
        # Flatten for overall distribution comparison
        pred_flat = predictions.flatten()
        actual_flat = actuals.flatten()
        
        # Remove any NaN or Inf values
        pred_flat = pred_flat[np.isfinite(pred_flat)]
        actual_flat = actual_flat[np.isfinite(actual_flat)]
        
        if len(pred_flat) == 0 or len(actual_flat) == 0:
            return 1.0
        
        # Create histograms
        min_val = min(np.min(pred_flat), np.min(actual_flat))
        max_val = max(np.max(pred_flat), np.max(actual_flat))
        
        # Handle case where all values are the same
        if min_val == max_val:
            return 0.0
            
        pred_hist, _ = np.histogram(pred_flat, bins=bins, range=(min_val, max_val), density=True)
        actual_hist, _ = np.histogram(actual_flat, bins=bins, range=(min_val, max_val), density=True)
        
        # Add small epsilon to avoid division by zero
        pred_hist = pred_hist + 1e-10
        actual_hist = actual_hist + 1e-10
        
        js_div = jensenshannon(pred_hist, actual_hist)
        return js_div
    
    def calculate_adversarial_robustness(self, predictions, actuals, noise_level=0.1):
        """Calculate Adversarial Robustness Score"""
        if predictions.size == 0 or actuals.size == 0:
            return 0.0
            
        original_predictions = predictions.copy()
        
        # Add small perturbations to inputs (simulating adversarial noise)
        noisy_predictions = predictions + np.random.normal(0, noise_level * np.std(predictions), predictions.shape)
        
        # Calculate change in predictions relative to actuals
        original_error = np.mean(np.abs(original_predictions - actuals))
        noisy_error = np.mean(np.abs(noisy_predictions - actuals))
        
        # Robustness score: how much the error increases with noise
        error_increase = noisy_error - original_error
        robustness_score = 1.0 / (1.0 + error_increase / (original_error + 1e-8))
        
        return min(max(robustness_score, 0.0), 1.0)  # Clip to [0, 1]
    
    def calculate_wasserstein_distance(self, predictions, actuals):
        """Calculate Wasserstein Distance (Earth Mover's Distance)"""
        if predictions.size == 0 or actuals.size == 0:
            return 1.0
            
        pred_flat = predictions.flatten()
        actual_flat = actuals.flatten()
        
        pred_flat = pred_flat[np.isfinite(pred_flat)]
        actual_flat = actual_flat[np.isfinite(actual_flat)]
        
        if len(pred_flat) == 0 or len(actual_flat) == 0:
            return 1.0
            
        return wasserstein_distance(pred_flat, actual_flat)
    
    def calculate_distribution_metrics(self, predictions, actuals):
        """Calculate comprehensive distribution metrics"""
        if predictions.size == 0 or actuals.size == 0:
            return {
                'ks_statistic': 1.0,
                'ks_p_value': 0.0,
                'js_divergence': 1.0,
                'wasserstein_distance': 1.0,
                'mean_absolute_error': float('inf'),
                'std_ratio': 1.0
            }
        
        pred_flat = predictions.flatten()
        actual_flat = actuals.flatten()
        
        pred_flat = pred_flat[np.isfinite(pred_flat)]
        actual_flat = actual_flat[np.isfinite(actual_flat)]
        
        if len(pred_flat) == 0 or len(actual_flat) == 0:
            return {
                'ks_statistic': 1.0,
                'ks_p_value': 0.0,
                'js_divergence': 1.0,
                'wasserstein_distance': 1.0,
                'mean_absolute_error': float('inf'),
                'std_ratio': 1.0
            }
        
        ks_stat, ks_p = self.calculate_ks_test(predictions, actuals)
        js_div = self.calculate_js_divergence(predictions, actuals)
        wasserstein_dist = self.calculate_wasserstein_distance(predictions, actuals)
        
        mae = np.mean(np.abs(pred_flat - actual_flat))
        std_ratio = np.std(pred_flat) / (np.std(actual_flat) + 1e-8)
        
        return {
            'ks_statistic': ks_stat,
            'ks_p_value': ks_p,
            'js_divergence': js_div,
            'wasserstein_distance': wasserstein_dist,
            'mean_absolute_error': mae,
            'std_ratio': std_ratio
        }
    
    def calculate_msmape(self, predictions, actuals):
        """Calculate M-SMAPE (Modified Symmetric Mean Absolute Percentage Error)"""
        if predictions.size == 0 or actuals.size == 0:
            return float('inf')
            
        epsilon = 1e-8
        numerator = np.abs(predictions - actuals)
        denominator = (np.abs(predictions) + np.abs(actuals)) / 2 + epsilon
        
        smape = 2 * np.mean(numerator / denominator)
        return smape * 100  # Return as percentage
    def calculate_mae(self, predictions, actuals):
        """Calculate Mean Absolute Error (MAE)"""
        if predictions.size == 0 or actuals.size == 0:
            return float('inf')
            
        mae = np.mean(np.abs(predictions - actuals))
        return mae
    
    def calculate_mape(self, predictions, actuals):
        """Calculate Mean Absolute Percentage Error (MAPE)"""
        if predictions.size == 0 or actuals.size == 0:
            return float('inf')
            
        # Add small epsilon to avoid division by zero
        epsilon = 1e-8
        mape = np.mean(np.abs((actuals - predictions) / (actuals + epsilon))) * 100
        return mape
    
    def calculate_rmse(self, predictions, actuals):
        """Calculate Root Mean Squared Error (RMSE)"""
        if predictions.size == 0 or actuals.size == 0:
            return float('inf')
            
        rmse = np.sqrt(np.mean((predictions - actuals) ** 2))
        return rmse
    
    def calculate_rrse(self, predictions, actuals):
        """Calculate Root Relative Squared Error (as in train_test.py)"""
        if predictions.size == 0 or actuals.size == 0:
            return float('inf')
            
        # Flatten arrays
        pred_flat = predictions.flatten()
        actual_flat = actuals.flatten()
        
        # RRSE calculation
        sum_squared_diff = np.sum((pred_flat - actual_flat) ** 2)
        root_sum_squared = np.sqrt(sum_squared_diff)
        
        mean_actual = np.mean(actual_flat)
        diff_r = actual_flat - mean_actual
        sum_squared_r = np.sum(diff_r ** 2)
        root_sum_squared_r = np.sqrt(sum_squared_r)
        
        rrse = root_sum_squared / root_sum_squared_r
        return rrse
    
    def calculate_rae(self, predictions, actuals):
        """Calculate Relative Absolute Error (as in train_test.py)"""
        if predictions.size == 0 or actuals.size == 0:
            return float('inf')
            
        # Flatten arrays
        pred_flat = predictions.flatten()
        actual_flat = actuals.flatten()
        
        # RAE calculation
        sum_absolute_diff = np.sum(np.abs(pred_flat - actual_flat))
        
        mean_actual = np.mean(actual_flat)
        sum_absolute_r = np.sum(np.abs(actual_flat - mean_actual))
        
        rae = sum_absolute_diff / sum_absolute_r
        return rae
    
    def calculate_auc_roc(self, predictions, actuals, threshold=0.3):
        """Calculate AUC-ROC for binary threat detection"""
        if predictions.size == 0 or actuals.size == 0:
            return 0.5, np.array([0, 1]), np.array([0, 1])
            
        # Convert to binary classification (threat vs no-threat)
        pred_binary = (predictions > threshold).astype(int)
        actual_binary = (actuals > threshold).astype(int)
        
        # Flatten for overall AUC
        pred_flat = pred_binary.flatten()
        actual_flat = actual_binary.flatten()
        
        try:
            auc = roc_auc_score(actual_flat, pred_flat)
            fpr, tpr, _ = roc_curve(actual_flat, pred_flat)
            return auc, fpr, tpr
        except:
            return 0.5, np.array([0, 1]), np.array([0, 1])  # Random classifier
    
    def run_comprehensive_evaluation(self):
        """Run all evaluation metrics for BMTGNN model"""
        print("Starting comprehensive BMTGNN model evaluation...")
        
        # Generate predictions
        print("\nGenerating training predictions...")
        pred_train, actual_train, var_train, ci_train, all_outputs_train = self.generate_predictions('train')
        
        print("\nGenerating testing predictions...")
        pred_test, actual_test, var_test, ci_test, all_outputs_test = self.generate_predictions('test')
        
        if pred_train.size == 0 or pred_test.size == 0:
            print("Warning: No predictions generated. Evaluation will use placeholder metrics.")
            self.initialize_placeholder_metrics()
            return self.metrics
        
        print(f"Training predictions: {pred_train.shape}, actuals: {actual_train.shape}")
        print(f"Testing predictions: {pred_test.shape}, actuals: {actual_test.shape}")
        
        # Calculate metrics
        threshold = 0.3  # Detection threshold
        
        # 1. Accuracy Metrics (BMTGNN specific)
        self.metrics['M-SMAPE_Train'] = self.calculate_msmape(pred_train, actual_train)
        self.metrics['M-SMAPE_Test'] = self.calculate_msmape(pred_test, actual_test)
        
        self.metrics['MAPE_Train'] = self.calculate_mape(pred_train, actual_train)
        self.metrics['MAPE_Test'] = self.calculate_mape(pred_test, actual_test)
        
        self.metrics['RMSE_Train'] = self.calculate_rmse(pred_train, actual_train)
        self.metrics['RMSE_Test'] = self.calculate_rmse(pred_test, actual_test)
        
        self.metrics['RRSE_Train'] = self.calculate_rrse(pred_train, actual_train)
        self.metrics['RRSE_Test'] = self.calculate_rrse(pred_test, actual_test)
        
        self.metrics['RAE_Train'] = self.calculate_rae(pred_train, actual_train)
        self.metrics['RAE_Test'] = self.calculate_rae(pred_test, actual_test)
        
        self.metrics['MAE_Train'] = self.calculate_mae(pred_train, actual_train)
        self.metrics['MAE_Test'] = self.calculate_mae(pred_test, actual_test)
        
        # 2. Detection Quality Metrics
        auc_train, fpr_curve_train, tpr_curve_train = self.calculate_auc_roc(pred_train, actual_train, threshold)
        auc_test, fpr_curve_test, tpr_curve_test = self.calculate_auc_roc(pred_test, actual_test, threshold)
        self.metrics['AUC_ROC_Train'] = auc_train
        self.metrics['AUC_ROC_Test'] = auc_test
        
        # 3. Robustness and Distribution Metrics
        print("Calculating robustness and distribution metrics...")
        dist_metrics_train = self.calculate_distribution_metrics(pred_train, actual_train)
        dist_metrics_test = self.calculate_distribution_metrics(pred_test, actual_test)
        
        self.metrics['KS_Statistic_Train'] = dist_metrics_train['ks_statistic']
        self.metrics['KS_Statistic_Test'] = dist_metrics_test['ks_statistic']
        self.metrics['KS_p_value_Train'] = dist_metrics_train['ks_p_value']
        self.metrics['KS_p_value_Test'] = dist_metrics_test['ks_p_value']
        self.metrics['JS_Divergence_Train'] = dist_metrics_train['js_divergence']
        self.metrics['JS_Divergence_Test'] = dist_metrics_test['js_divergence']
        self.metrics['Wasserstein_Distance_Train'] = dist_metrics_train['wasserstein_distance']
        self.metrics['Wasserstein_Distance_Test'] = dist_metrics_test['wasserstein_distance']
        
        # Adversarial Robustness
        self.metrics['Adversarial_Robustness_Train'] = self.calculate_adversarial_robustness(pred_train, actual_train)
        self.metrics['Adversarial_Robustness_Test'] = self.calculate_adversarial_robustness(pred_test, actual_test)
        
        # 4. Quantile Loss (Uncertainty Quantification)
        print("Calculating quantile losses...")
        quantile_loss_train = self.calculate_quantile_loss(all_outputs_train, actual_train)
        quantile_loss_test = self.calculate_quantile_loss(all_outputs_test, actual_test)
        
        self.metrics.update({f'Quantile_Loss_Train_{k}': v for k, v in quantile_loss_train.items()})
        self.metrics.update({f'Quantile_Loss_Test_{k}': v for k, v in quantile_loss_test.items()})
        
        # 5. Store additional data for visualization
        self.evaluation_data = {
            'predictions_train': pred_train,
            'actuals_train': actual_train,
            'predictions_test': pred_test,
            'actuals_test': actual_test,
            'variances_train': var_train,
            'variances_test': var_test,
            'confidence_intervals_train': ci_train,
            'confidence_intervals_test': ci_test,
            'all_outputs_train': all_outputs_train,
            'all_outputs_test': all_outputs_test,
            'fpr_curve_train': fpr_curve_train,
            'tpr_curve_train': tpr_curve_train,
            'fpr_curve_test': fpr_curve_test,
            'tpr_curve_test': tpr_curve_test,
            'distribution_metrics_train': dist_metrics_train,
            'distribution_metrics_test': dist_metrics_test
        }
        
        return self.metrics


    def plot_bmtgnn_analysis(self):
        """Create detailed analysis plots for BMTGNN model including robustness and quantile charts"""
        if not hasattr(self, 'evaluation_data') or self.evaluation_data['predictions_test'].size == 0:
            print("No evaluation data available for plotting")
            return
            
        # Create a 3x2 subplot grid for comprehensive analysis
        fig, axes = plt.subplots(3, 2, figsize=(20, 18))
        
        # 1. ROC Curves
        axes[0, 0].plot(self.evaluation_data['fpr_curve_train'], 
                       self.evaluation_data['tpr_curve_train'], 
                       'b-', linewidth=2, label=f'Train (AUC = {self.metrics["AUC_ROC_Train"]:.3f})')
        axes[0, 0].plot(self.evaluation_data['fpr_curve_test'], 
                       self.evaluation_data['tpr_curve_test'], 
                       'r-', linewidth=2, label=f'Test (AUC = {self.metrics["AUC_ROC_Test"]:.3f})')
        axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Classifier')
        axes[0, 0].set_xlabel('False Positive Rate')
        axes[0, 0].set_ylabel('True Positive Rate')
        axes[0, 0].set_title('BMTGNN: ROC Curves')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Distribution Comparison (KS Test)
        if self.evaluation_data['predictions_test'].size > 0:
            test_pred_flat = self.evaluation_data['predictions_test'].flatten()
            test_actual_flat = self.evaluation_data['actuals_test'].flatten()
            
            # Remove outliers for better visualization
            q1_pred, q3_pred = np.percentile(test_pred_flat, [25, 75])
            iqr_pred = q3_pred - q1_pred
            pred_mask = (test_pred_flat >= q1_pred - 1.5*iqr_pred) & (test_pred_flat <= q3_pred + 1.5*iqr_pred)
            
            q1_actual, q3_actual = np.percentile(test_actual_flat, [25, 75])
            iqr_actual = q3_actual - q1_actual
            actual_mask = (test_actual_flat >= q1_actual - 1.5*iqr_actual) & (test_actual_flat <= q3_actual + 1.5*iqr_actual)
            
            mask = pred_mask & actual_mask
            test_pred_filtered = test_pred_flat[mask]
            test_actual_filtered = test_actual_flat[mask]
            
            # Plot histograms
            axes[0, 1].hist(test_actual_filtered, bins=50, alpha=0.7, label='Actual', density=True, color='blue')
            axes[0, 1].hist(test_pred_filtered, bins=50, alpha=0.7, label='Predicted', density=True, color='red')
            axes[0, 1].set_xlabel('Values')
            axes[0, 1].set_ylabel('Density')
            axes[0, 1].set_title(f'Distribution Comparison (KS Stat: {self.metrics["KS_Statistic_Test"]:.3f})')
            axes[0, 1].legend()
            axes[0, 1].grid(True, alpha=0.3)
        
        # 3. Robustness Metrics Comparison
        robustness_metrics = ['Adversarial_Robustness_Train', 'Adversarial_Robustness_Test',
                            'JS_Divergence_Train', 'JS_Divergence_Test']
        robustness_values = [self.metrics[m] for m in robustness_metrics]
        
        x_robust = np.arange(len(robustness_metrics))
        bars = axes[1, 0].bar(x_robust, robustness_values, alpha=0.7)
        
        # Color bars based on metric type
        colors = ['green', 'lightgreen', 'orange', 'lightcoral']
        for i, bar in enumerate(bars):
            bar.set_color(colors[i])
        
        axes[1, 0].set_xticks(x_robust)
        axes[1, 0].set_xticklabels(robustness_metrics, rotation=45, ha='right')
        axes[1, 0].set_title('BMTGNN: Robustness and Distribution Metrics')
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. Quantile Loss Comparison
        quantile_metrics = ['Quantile_Loss_Train_quantile_10', 'Quantile_Loss_Train_quantile_50', 
                          'Quantile_Loss_Train_quantile_90', 'Quantile_Loss_Test_quantile_10',
                          'Quantile_Loss_Test_quantile_50', 'Quantile_Loss_Test_quantile_90']
        quantile_values = [self.metrics[m] if self.metrics[m] != float('inf') else 0 for m in quantile_metrics]
        
        x_quantile = np.arange(len(quantile_metrics))
        bars = axes[1, 1].bar(x_quantile, quantile_values, alpha=0.7)
        
        # Color bars
        colors = ['lightblue', 'blue', 'darkblue', 'lightcoral', 'red', 'darkred']
        for i, bar in enumerate(bars):
            bar.set_color(colors[i])
        
        axes[1, 1].set_xticks(x_quantile)
        axes[1, 1].set_xticklabels(['Q10_T', 'Q50_T', 'Q90_T', 'Q10_Te', 'Q50_Te', 'Q90_Te'], rotation=45)
        axes[1, 1].set_ylabel('Quantile Loss')
        axes[1, 1].set_title('BMTGNN: Quantile Loss (Pinball Loss) Comparison')
        axes[1, 1].grid(True, alpha=0.3)
        
        # 5. Prediction vs Actual Scatter (first node)
        if self.evaluation_data['predictions_test'].size > 0:
            test_pred_flat = self.evaluation_data['predictions_test'][:, :, 0].flatten()
            test_actual_flat = self.evaluation_data['actuals_test'][:, :, 0].flatten()
            
            # Take a sample for cleaner visualization
            sample_size = min(1000, len(test_pred_flat))
            indices = np.random.choice(len(test_pred_flat), sample_size, replace=False)
            
            axes[2, 0].scatter(test_actual_flat[indices], test_pred_flat[indices], 
                              alpha=0.5, s=10, color='blue')
            axes[2, 0].plot([test_actual_flat.min(), test_actual_flat.max()], 
                           [test_actual_flat.min(), test_actual_flat.max()], 'r--', alpha=0.8, linewidth=2)
            axes[2, 0].set_xlabel('Actual Values (Node 0)')
            axes[2, 0].set_ylabel('Predicted Values (Node 0)')
            axes[2, 0].set_title('BMTGNN: Prediction vs Actual (Test Set)')
            axes[2, 0].grid(True, alpha=0.3)
        
        # 6. Uncertainty Visualization (Quantile Ranges)
        if (hasattr(self, 'evaluation_data') and 
            'all_outputs_test' in self.evaluation_data and 
            self.evaluation_data['all_outputs_test'].size > 0):
            
            # Take first sequence for visualization
            first_seq_preds = self.evaluation_data['all_outputs_test'][:, 0, :, 0]  # (num_runs, seq_len)
            first_seq_actual = self.evaluation_data['actuals_test'][0, :, 0]  # (seq_len,)
            
            time_steps = range(first_seq_actual.shape[0])
            
            # Calculate quantiles across runs
            quantile_10 = np.percentile(first_seq_preds, 10, axis=0)
            quantile_50 = np.percentile(first_seq_preds, 50, axis=0)
            quantile_90 = np.percentile(first_seq_preds, 90, axis=0)
            
            axes[2, 1].fill_between(time_steps, quantile_10, quantile_90, alpha=0.3, color='orange', label='80% Confidence')
            axes[2, 1].plot(time_steps, quantile_50, 'r-', linewidth=2, label='Median Prediction')
            axes[2, 1].plot(time_steps, first_seq_actual, 'b-', linewidth=2, label='Actual')
            axes[2, 1].set_xlabel('Time Steps')
            axes[2, 1].set_ylabel('Values')
            axes[2, 1].set_title('BMTGNN: Uncertainty Quantification (First Sequence)')
            axes[2, 1].legend()
            axes[2, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('bmtgnn_comprehensive_analysis.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("BMTGNN comprehensive analysis plot saved as 'bmtgnn_comprehensive_analysis.png'")

    def generate_bmtgnn_report(self):
        """Generate comprehensive text report for BMTGNN model"""
        report_lines = []
        report_lines.append("=" * 80)
        report_lines.append("BMTGNN CYBER THREAT FORECASTING MODEL EVALUATION REPORT")
        report_lines.append("=" * 80)
        report_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report_lines.append(f"Model: {self.model_path}")
        report_lines.append(f"Data: {self.data_file}")
        report_lines.append("")
        
        # Model Configuration
        report_lines.append("BMTGNN MODEL CONFIGURATION")
        report_lines.append("-" * 40)
        for key, value in self.config.items():
            report_lines.append(f"{key}: {value}")
        report_lines.append("")
        
        # Executive Summary
        report_lines.append("EXECUTIVE SUMMARY")
        report_lines.append("-" * 40)
        
        # Calculate overall scores
        if self.metrics['M-SMAPE_Test'] != float('inf'):
            avg_msmape = (self.metrics['M-SMAPE_Train'] + self.metrics['M-SMAPE_Test']) / 2
            report_lines.append(f"Average M-SMAPE: {avg_msmape:.2f}%")
        else:
            report_lines.append("Average M-SMAPE: N/A (no predictions)")
            
        avg_auc = (self.metrics['AUC_ROC_Train'] + self.metrics['AUC_ROC_Test']) / 2
        report_lines.append(f"Average AUC-ROC: {avg_auc:.3f}")
        report_lines.append("")
        
        # Detailed Metrics
        report_lines.append("DETAILED METRICS")
        report_lines.append("-" * 40)
        
        # Accuracy Metrics
        report_lines.append("Accuracy Metrics:")
        accuracy_metrics = ['M-SMAPE_Train', 'M-SMAPE_Test', 'RRSE_Train', 'RRSE_Test', 'RAE_Train', 'RAE_Test', 
                            'MAPE_Train', 'MAPE_Test', 'MAE_Train', 'MAE_Test', 'RMSE_Train', 'RMSE_Test']
        for metric in accuracy_metrics:
            value = self.metrics[metric]
            if value == float('inf'):
                report_lines.append(f"  {metric}: N/A")
            elif 'SMAPE' in metric:
                report_lines.append(f"  {metric}: {value:.2f}%")
            else:
                report_lines.append(f"  {metric}: {value:.4f}")
        
        # Robustness and Distribution Metrics
        report_lines.append("\nRobustness and Distribution Metrics:")
        robustness_metrics = ['KS_Statistic_Train', 'KS_Statistic_Test', 
                            'KS_p_value_Train', 'KS_p_value_Test',
                            'JS_Divergence_Train', 'JS_Divergence_Test',
                            'Adversarial_Robustness_Train', 'Adversarial_Robustness_Test']
        for metric in robustness_metrics:
            value = self.metrics[metric]
            if 'p_value' in metric:
                report_lines.append(f"  {metric}: {value:.4f}")
            else:
                report_lines.append(f"  {metric}: {value:.3f}")
        
        # Uncertainty Quantification (Quantile Loss)
        report_lines.append("\nUncertainty Quantification (Quantile Loss):")
        quantile_metrics = ['Quantile_Loss_Train_quantile_10', 'Quantile_Loss_Train_quantile_50', 
                          'Quantile_Loss_Train_quantile_90', 'Quantile_Loss_Test_quantile_10',
                          'Quantile_Loss_Test_quantile_50', 'Quantile_Loss_Test_quantile_90']
        for metric in quantile_metrics:
            value = self.metrics[metric]
            if value == float('inf'):
                report_lines.append(f"  {metric}: N/A")
            else:
                report_lines.append(f"  {metric}: {value:.4f}")
        
        report_lines.append("")
        
        # Performance Assessment
        report_lines.append("PERFORMANCE ASSESSMENT")
        report_lines.append("-" * 40)
        
        # Assess model performance
        if avg_auc > 0.8:
            auc_assessment = "EXCELLENT"
        elif avg_auc > 0.7:
            auc_assessment = "GOOD"
        elif avg_auc > 0.6:
            auc_assessment = "FAIR"
        else:
            auc_assessment = "POOR"
        
        if self.metrics['M-SMAPE_Test'] != float('inf') and self.metrics['M-SMAPE_Test'] < 20:
            smape_assessment = "GOOD"
        elif self.metrics['M-SMAPE_Test'] != float('inf') and self.metrics['M-SMAPE_Test'] < 40:
            smape_assessment = "FAIR"
        else:
            smape_assessment = "NEEDS IMPROVEMENT"
        
        # Robustness assessment
        if self.metrics['Adversarial_Robustness_Test'] > 0.9:
            robustness_assessment = "EXCELLENT"
        elif self.metrics['Adversarial_Robustness_Test'] > 0.7:
            robustness_assessment = "GOOD"
        else:
            robustness_assessment = "NEEDS IMPROVEMENT"
        
        report_lines.append(f"Detection Quality: {auc_assessment} (AUC-ROC: {avg_auc:.3f})")
        if self.metrics['M-SMAPE_Test'] != float('inf'):
            report_lines.append(f"Forecast Accuracy: {smape_assessment} (M-SMAPE: {self.metrics['M-SMAPE_Test']:.2f}%)")
        else:
            report_lines.append(f"Forecast Accuracy: N/A (no predictions generated)")
        report_lines.append(f"Model Robustness: {robustness_assessment} (Score: {self.metrics['Adversarial_Robustness_Test']:.3f})")
        
        # Distribution fit assessment
        if self.metrics['KS_Statistic_Test'] < 0.1:
            dist_assessment = "EXCELLENT FIT"
        elif self.metrics['KS_Statistic_Test'] < 0.2:
            dist_assessment = "GOOD FIT"
        else:
            dist_assessment = "POOR DISTRIBUTION MATCH"
        
        report_lines.append(f"Distribution Fit: {dist_assessment} (KS: {self.metrics['KS_Statistic_Test']:.3f})")
        
        # Recommendations
        report_lines.append("\nRECOMMENDATIONS")
        report_lines.append("-" * 40)
        
        if self.metrics['KS_Statistic_Test'] > 0.2:
            report_lines.append("POOR DISTRIBUTION MATCH: Consider model calibration or different architecture")
        if self.metrics['JS_Divergence_Test'] > 0.3:
            report_lines.append("HIGH DISTRIBUTION DIVERGENCE: Review feature engineering and normalization")
        if self.metrics['Adversarial_Robustness_Test'] < 0.7:
            report_lines.append("LOW ROBUSTNESS: Add regularization or adversarial training")
        
        if (self.metrics['KS_Statistic_Test'] < 0.15 and 
            self.metrics['Adversarial_Robustness_Test'] > 0.8):
            report_lines.append("STRONG MODEL: Current configuration shows good performance and robustness")
        
        report_lines.append("")
        report_lines.append("=" * 80)
        
        # Write report to file
        report_path = "bmtgnn_comprehensive_report.txt"
        with open(report_path, "w", encoding="utf-8") as f:
            f.write('\n'.join(report_lines))
        
        print(f"BMTGNN comprehensive report saved to: {report_path}")
        return report_path

    def initialize_placeholder_metrics(self):
        """Initialize metrics with placeholder values when no predictions are generated"""
        placeholder_metrics = {
            'M-SMAPE_Train': float('inf'),
            'M-SMAPE_Test': float('inf'),
            'RRSE_Train': float('inf'),
            'RRSE_Test': float('inf'),
            'RAE_Train': float('inf'),
            'RAE_Test': float('inf'),
            'AUC_ROC_Train': 0.5,
            'AUC_ROC_Test': 0.5,
            'KS_Statistic_Train': 1.0,
            'KS_Statistic_Test': 1.0,
            'KS_p_value_Train': 0.0,
            'KS_p_value_Test': 0.0,
            'JS_Divergence_Train': 1.0,
            'JS_Divergence_Test': 1.0,
            'Wasserstein_Distance_Train': 1.0,
            'Wasserstein_Distance_Test': 1.0,
            'Adversarial_Robustness_Train': 0.0,
            'Adversarial_Robustness_Test': 0.0,
            'Quantile_Loss_Train_quantile_10': float('inf'),
            'Quantile_Loss_Train_quantile_50': float('inf'),
            'Quantile_Loss_Train_quantile_90': float('inf'),
            'Quantile_Loss_Test_quantile_10': float('inf'),
            'Quantile_Loss_Test_quantile_50': float('inf'),
            'Quantile_Loss_Test_quantile_90': float('inf'),
        }
        self.metrics.update(placeholder_metrics)

def main():
    """Main execution function for BMTGNN evaluation"""
    # Configuration - adjust these paths to match your BMTGNN setup
    model_path = "model/Bayesian/o_model.pt"  # Path to your trained BMTGNN model
    data_file = "./data/sm_data.txt"  # Path to your data file
    config_file = "model/Bayesian/hp.txt"  # Path to hyperparameters file
    
    # Initialize evaluator
    evaluator = BMTGNNEvaluator(
        model_path=model_path,
        data_file=data_file,
        config_file=config_file
    )
    
    # Run comprehensive evaluation
    metrics = evaluator.run_comprehensive_evaluation()
    
    # Generate visualizations
    evaluator.plot_bmtgnn_analysis()
    
    # Generate text report
    report_path = evaluator.generate_bmtgnn_report()
    
    print("\nBMTGNN Comprehensive Evaluation completed successfully!")
    print(f"Key metrics calculated: {len(metrics)}")
    print(f"Report generated: {report_path}")
    
    # Print key metrics to console
    print("\nKEY BMTGNN METRICS SUMMARY:")
    print(f"M-SMAPE Test: {metrics.get('M-SMAPE_Test', 'N/A'):.2f}%")
    print(f"RRSE Test: {metrics.get('RRSE_Test', 'N/A'):.4f}")
    print(f"AUC-ROC Test: {metrics.get('AUC_ROC_Test', 'N/A'):.3f}")
    print(f"KS Statistic Test: {metrics.get('KS_Statistic_Test', 'N/A'):.3f}")
    print(f"JS Divergence Test: {metrics.get('JS_Divergence_Test', 'N/A'):.3f}")
    print(f"Adversarial Robustness Test: {metrics.get('Adversarial_Robustness_Test', 'N/A'):.3f}")
    print(f"Quantile Loss Q50 Test: {metrics.get('Quantile_Loss_Test_quantile_50', 'N/A'):.4f}")

if __name__ == "__main__":
    main()

In [ ]:
"""4. Script to forecast future time points using the final model trained on the full data."""

import pickle
import numpy as np
import os
import scipy.sparse as sp
import torch
from scipy.sparse import linalg
from torch.autograd import Variable
import sys
import csv
from collections import defaultdict
from matplotlib import pyplot
import random

pyplot.rcParams['savefig.dpi'] = 1200


def exponential_smoothing(series, alpha):

    result = [series[0]] # first value is same as series
    for n in range(1, len(series)):
        result.append(alpha * series[n] + (1 - alpha) * result[n-1])
    return result

def consistent_name(name):

    name=name.replace('-ALL','').replace('Mentions-','').replace(' ALL','').replace('Solution_','').replace('_Mentions','')
    
    #special case
    if 'HIDDEN MARKOV MODEL' in name:
        return 'Statistical HMM'

    if name=='CAPTCHA' or name=='DNSSEC' or name=='RRAM':
        return name

    if 'IZ' in name:
        name=name.replace('IZ', 'IS')# applicable only in our data (British English)
    if 'IOR' in name:
        name=name.replace('IOR','IOUR')#behaviour (British English)

    #e.g., University of london
    if not name.isupper():
        words=name.split(' ')
        result=''
        for i,word in enumerate(words):
            if len(word)<=2: #e.g., "of"
                result+=word
            else:
                result+=word[0].upper()+word[1:]
            
            if i<len(words)-1:
                result+=' '

        return result
    

    words= name.split(' ')
    result=''
    for i,word in enumerate(words):
        if len(word)<=3 or '/' in word or word=='MITM' or word =='SIEM':
            result+=word
        else:
            result+=word[0]+(word[1:].lower())
        
        if i<len(words)-1:
            result+=' '
        
    return result

#returns the closest curve cc to a given curve c in a list of forecasted curves, where cc is strictly larger than c
def getClosestCurveLarger(c,forecast,confidence, attack, solutions,col):
    d=999999999
    cc=None
    cc_conf=None
    for j in range(forecast.shape[1]):
        f= forecast[:,j]
        f_conf=confidence[:,j]
        if not col[j] in solutions and not col[j]==attack: #exclude irrelevant curves
            continue 
        if torch.mean(f) <= torch.mean(c):
            continue #must be larger
        if torch.mean(f)-torch.mean(c)<d:
            d=torch.mean(f)-torch.mean(c)
            cc=f.clone()
            cc_conf=f_conf.clone()
    return cc,cc_conf

#returns closest curve cc to given curve c in a list of forecasted curves, where cc is strictly smaller than c
def getClosestCurveSmaller(c,forecast,confidence,attack, solutions, col):
    d=999999999
    cc=None
    cc_conf=None
    for j in range(forecast.shape[1]):
        f= forecast[:,j]
        f_conf=confidence[:,j]
        if not col[j] in solutions and not col[j]==attack: #exclude irrelevant curves
            continue 
        if torch.mean(f) >= torch.mean(c):
            continue #must be smaller
        if torch.abs(torch.mean(f)-torch.mean(c))<d:
            d=torch.abs(torch.mean(f)-torch.mean(c))
            cc=f.clone()
            cc_conf=f_conf.clone()
    return cc,cc_conf

#negative values (due to smoothing) are changed to 0
def zero_negative_curves(data, forecast, attack, solutions, index, col):

    # --- Helper: normalize names like in plot_forecast() ---
    def normalize(name):
        return name.strip().replace('/', '_').replace('\\', '_').replace(' ', '_')

    # Ensure everything is normalized consistently
    col = [normalize(c) for c in col]
    if isinstance(attack, str):
        attack = [attack]
    attack = [normalize(a) for a in attack]
    solutions = [normalize(s) for s in solutions]

    # Normalize index keys
    if isinstance(index, dict):
        index = {normalize(k): v for k, v in index.items()}

    valid_col_count = data.shape[1]
    print(f"[DEBUG] zero_negative_curves: {valid_col_count} columns available")

    # --- Ensure all indices are within data.shape bounds ---
    safe_index = {k: v for k, v in index.items() if v < valid_col_count}
    dropped = set(index.keys()) - set(safe_index.keys())
    if dropped:
        print(f"[WARNING] Dropping out-of-range indices: {dropped}")
    index = safe_index

    # --- Process each attack ---
    for atk in attack:
        if atk not in index:
            print(f"[WARNING] Skipping unmatched attack: {atk}")
            continue
        i = index[atk]
        a = data[:, i]
        f = forecast[:, i]

        # Zero out negative values safely
        a[a < 0] = 0
        f[f < 0] = 0

        data[:, i] = a
        forecast[:, i] = f

    # --- Process each solution ---
    for s in solutions:
        if s not in index:
            print(f"[WARNING] Skipping unmatched solution: {s}")
            continue
        i = index[s]
        a = data[:, i]
        f = forecast[:, i]

        # Zero out negative values safely
        a[a < 0] = 0
        f[f < 0] = 0

        data[:, i] = a
        forecast[:, i] = f

    print(f"[INFO] zero_negative_curves complete: "
          f"{len(attack)} attacks | {len(solutions)} solutions processed")

    return data, forecast

    
    
           
        

#plots forecast of attack and relevant solutions trends. If alarming is set to True, plots the solutions trend forecasted to be less than the attack trend.
def plot_forecast(data, forecast, confidence, attack, solutions, index, col, alarming=True):

    # --- Helper: Normalize names consistently ---
    def normalize(name):
        return name.strip().replace('/', '_').replace('\\', '_').replace(' ', '_')

    # Normalize everything to prevent mismatched keys
    col = [normalize(c) for c in col]
    if isinstance(attack, str):
        attack = [attack]
    attack = [normalize(a) for a in attack]
    solutions = [normalize(s) for s in solutions]

    # Normalize index dictionary keys
    if isinstance(index, dict):
        index = {normalize(k): v for k, v in index.items()}

    print(f"[INFO] Available columns: {len(col)} | Normalized index keys: {len(index)}")

    # --- Defensive call to zero_negative_curves() ---
    data, forecast = zero_negative_curves(data, forecast, attack, solutions, index, col)

    # --- Plot setup ---
    colours = [
        "RoyalBlue", "Crimson", "DarkOrange", "MediumPurple", "MediumVioletRed",
        "DodgerBlue", "Indigo", "coral", "hotpink", "DarkMagenta",
        "SteelBlue", "brown", "MediumAquamarine", "SlateBlue", "SeaGreen",
        "MediumSpringGreen", "DarkOliveGreen", "Teal", "OliveDrab", "MediumSeaGreen",
        "DeepSkyBlue", "MediumSlateBlue", "MediumTurquoise", "FireBrick",
        "DarkCyan", "violet", "MediumOrchid", "DarkSalmon", "DarkRed"
    ]

    fig = pyplot.figure()
    ax = fig.add_axes([0.1, 0.1, 0.7, 0.75])
    counter = 0

    # --- Plot each attack ---
    for atk in attack:
        if atk not in index:
            print(f"⚠️ Skipping unmatched attack: {atk}")
            continue

        # Join past data and forecast
        d = torch.cat((data[:, index[atk]], forecast[0:1, index[atk]]), dim=0)
        f = forecast[:, index[atk]]
        c = confidence[:, index[atk]]
        label = atk

        ax.plot(range(len(d)), d, '-', color=colours[counter], label=label, linewidth=2)
        ax.plot(range(len(d)-1, (len(d)+len(f))-1), f, '-', color=colours[counter], linewidth=2)
        ax.fill_between(
            range(len(d)-1, (len(d)+len(f))-1),
            f - c, f + c,
            color=colours[counter], alpha=0.6
        )

        f_attack = f.clone()
        counter += 1

    # --- Remove irrelevant solutions if alarming mode is on ---
    if alarming and 'f_attack' in locals():
        solutions = [
            s for s in solutions
            if s in index and torch.mean(forecast[:, index[s]]) < torch.mean(f_attack)
        ]

    # --- Plot solutions ---
    for s in solutions:
        if s not in index:
            print(f"⚠️ Skipping unmatched solution: {s}")
            continue

        d = torch.cat((data[:, index[s]], forecast[0:1, index[s]]), dim=0)
        f = forecast[:, index[s]]
        c = confidence[:, index[s]]
        label = s

        ax.plot(range(len(d)), d, '-', color=colours[counter], label=label, linewidth=1)
        ax.plot(range(len(d)-1, (len(d)+len(f))-1), f, '-', color=colours[counter], linewidth=1)
        ax.fill_between(
            range(len(d)-1, (len(d)+len(f))-1),
            f - c, f + c,
            color=colours[counter], alpha=0.6
        )

        # Highlight difference between attack and solution trends
        if torch.mean(f_attack) > torch.mean(f):
            cc, cc_conf = getClosestCurveLarger(f, forecast, confidence, attack, solutions, col)
            ax.fill_between(
                range(len(d)-1, (len(d)+len(f))-1),
                cc - cc_conf, f + c,
                color=colours[counter], alpha=0.3
            )
        else:
            cc, cc_conf = getClosestCurveSmaller(f, forecast, confidence, attack, solutions, col)
            ax.fill_between(
                range(len(d)-1, (len(d)+len(f))-1),
                cc + cc_conf, f - c,
                color=colours[counter], alpha=0.3
            )
        counter += 1

    # --- X-axis, labels, etc. ---
    years = ['2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019',
             '2020', '2021', '2022', '2023', '2024', '2025', '2026']
    ax.set_xticks([6, 18, 30, 42, 54, 66, 78, 90, 102, 114, 126, 138, 150, 162, 174], years)
    ax.set_ylabel("Trend", fontsize=15)
    pyplot.yticks(fontsize=13)
    ax.legend(loc="upper left", prop={'size': 10}, bbox_to_anchor=(1, 1.03))
    ax.axis('tight')
    ax.grid(True)
    pyplot.xticks(rotation=90, fontsize=13)
    pyplot.title("Forecast Plot", y=1.03, fontsize=18)

    # --- Save & show ---
    fig = pyplot.gcf()
    fig.set_size_inches(10, 7)
    images_dir = 'model/Bayesian/forecast/plots/'
    first_attack = attack[0] if len(attack) > 0 else "unknown_attack"
    pyplot.savefig(images_dir + first_attack + '.png', bbox_inches="tight")
    pyplot.savefig(images_dir + first_attack + ".pdf", bbox_inches="tight", format='pdf')
    pyplot.show(block=False)
    pyplot.pause(5)
    pyplot.close()


#saves the numerical forecast to text file as well as past data of each node
def save_data(data, forecast, confidence, variance, col):
    # write the data and forecast
    for i in range(data.shape[1]):
        d= data[:,i]
        f= forecast[:,i]
        c=confidence[:,i]
        v=variance[:,i]
        name=col[i]
        file_dir = 'model/Bayesian/forecast/data/'
        with open(file_dir+name.replace('/','_')+'.txt', 'w') as ff:
            ff.write('Data: '+str(d.tolist())+'\n')
            ff.write('Forecast: '+str(f.tolist())+'\n')
            ff.write('95% Confidence: '+str(c.tolist())+'\n')
            ff.write('Variance: '+str(v.tolist())+'\n')
    ff.close()

#saves the forecasted trend's gap between attack and its relevant solutions to a csv file. The gap is for 3 years resulting in 3 values per solution.
def save_gap(forecast, attack, solutions,index):
    # write the data and forecast
    with open('model/Bayesian/forecast/gap/'+consistent_name(attack).replace('/','_')+'_gap.csv', 'w', newline='') as file:
        writer = csv.writer(file)
        # Write the list as a row
        writer.writerow(['Solution','2023','2024','2025'])
        table=[]
        a=forecast[:,index[attack]].tolist()
        a_reduced= [sum(a[i:i+12]) / 12 for i in range(0, len(a), 12)]#mean of every 12 months
        for s in solutions:
            row=[consistent_name(s)]
            f=forecast[:,index[s]].tolist()
            f_reduced= [sum(f[i:i+12]) / 12 for i in range(0, len(f), 12)]#mean of every 12 months
            
            gap=[x - y for x, y in zip(a_reduced, f_reduced)]#calculate the gap
            row.extend(gap)#3 years gap
            table.append(row)
        sorted_table = sorted(table, key=lambda row: sum(row[-3:]))
        for row in sorted_table:
            writer.writerow(row)
    





#given data file, returns the list of column names and dictionary of the format (column name,column index)
def create_columns(file_name):
    col_name = []
    col_index = {}

    with open(file_name, 'r') as f:
        reader = csv.reader(f)
        col_name = [c.strip().replace('/', '_') for c in next(reader)]
        if 'Date' in col_name[0]:
            col_name = col_name[1:]
        
        for i, c in enumerate(col_name):
            col_index[c] = i
        
    return col_name, col_index




#builds the attacks and pertinent technologies graph
def build_graph(file_name):
    # Initialise an empty dictionary with default value as an empty list
    graph = defaultdict(list)

    # Read the graph CSV file
    with open(file_name, 'r') as f:
        reader = csv.reader(f)
        # Iterate over each row in the CSV file
        for row in reader:
            # Extract the key node from the first column
            key_node = row[0]
            # Extract the adjacent nodes from the remaining columns
            adjacent_nodes =  [node for node in row[1:] if node]#does not include empty columns
            
            # Add the adjacent nodes to the graph dictionary
            graph[key_node].extend(adjacent_nodes)
    print('Graph loaded with',len(graph),'attacks...')
    return graph




#This script forecasts the future of the graph, up to 3 years in advance


data_file='./data/sm_data.txt'
model_file='model/Bayesian/o_model.pt'
nodes_file='data/data.csv'
graph_file='data/graph.csv'


#read the data
fin = open(data_file)
rawdat = np.loadtxt(fin, delimiter='\t')
n, m = rawdat.shape

#load column names and dictionary of (column name, index)
col,index=create_columns(nodes_file)


#build the graph in the format {attack:list of pertinent technologies}
graph=build_graph(graph_file)

#for normalisation
scale = np.ones(m)
dat = np.zeros(rawdat.shape)

#normalise
for i in range(m):
    scale[i] = np.max(np.abs(rawdat[:, i]))
    dat[:, i] = rawdat[:, i] / np.max(np.abs(rawdat[:, i]))
    

print('data shape:',dat.shape)

#preparing last part of the data to be used for the forecast
P=10 #look back
X= torch.from_numpy(dat[-P:, :]) #look back 10 months
X = torch.unsqueeze(X,dim=0)
X = torch.unsqueeze(X,dim=1)
X = X.transpose(2,3)
X = X.to(torch.float)



#load the model
model=None
with open(model_file, 'rb') as f:
    model = torch.load(f, map_location=torch.device("cpu"), weights_only=False)


# Bayesian estimation
num_runs = 10

# Create a list to store the outputs
outputs = []


# Use model to predict next time step
for _ in range(num_runs):
    with torch.no_grad():
        output = model(X)  
        y_pred = output[-1, :, :,-1].clone()#36x142
    outputs.append(y_pred)

# Stack the outputs along a new dimension
outputs = torch.stack(outputs)


Y=torch.mean(outputs,dim=0)
variance = torch.var(outputs, dim=0)#variance
std_dev = torch.std(outputs, dim=0)#standard deviation
# Calculate 95% confidence interval
z=1.96
confidence=z*std_dev/torch.sqrt(torch.tensor(num_runs))

dat*=scale
Y*=scale
variance*=scale
confidence*=scale


print('output shape:',Y.shape)

#----------------------------------------------------------------------------------------------------#

#Plotting:



#save the data to desk
dat=torch.from_numpy(dat)
save_data(dat,Y,confidence,variance,col)

#combine data
all=torch.cat((dat,Y), dim=0)


#scale down full data (global normalisation)
incident_max=-999999999
mention_max=-999999999

for i in range(all.shape[0]):
    for j in range(all.shape[1]):
        if 'WAR' in col[j] or 'Holiday' in col[j] or j in range(16,32):
            continue
        if 'Mention' in col[j]:
            if all[i,j]>mention_max:
                mention_max=all[i,j]
        else:
            if all[i,j]>incident_max:
                incident_max=all[i,j]

all_n=torch.zeros(all.shape[0],all.shape[1])
confidence_n=torch.zeros(confidence.shape[0],confidence.shape[1])
u=0
for i in range(all.shape[0]):
    for j in range(all.shape[1]):
            if 'Mention' in col[j]:
                all_n[i,j]=all[i,j]/mention_max
            else:
                all_n[i,j]=all[i,j]/incident_max
            
            if i>=all.shape[0]-36:
                confidence_n[u,j]=confidence[u,j]*(all_n[i,j]/all[i,j])
    if i>=all.shape[0]-36:
        u+=1


#smoothing
smoothed_dat=torch.stack(exponential_smoothing(all_n, 0.1))
smoothed_confidence=torch.stack(exponential_smoothing(confidence_n, 0.1))





#plot all forecasted nodes in the graph as groups of plots. Each plot consists of a single attack and its pertinent technologies
for attack, solutions in graph.items():
    plot_forecast(smoothed_dat[:-36,],smoothed_dat[-36:,], smoothed_confidence,attack,solutions,index,col)
    save_gap(smoothed_dat[-36:,],attack,solutions,index) #save gaps of each attack to file
    




